In [1]:
# Step 0: Imports and setup
# Description: Load necessary libraries, define file paths, and timestamp for outputs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

# File paths
input_file = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_priority_index_step8.csv"
output_folder_base = "/Users/jakegehrung/Desktop/data_raw/final_deliverables"

# Timestamp for unique file/folder names
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_excel = os.path.join(output_folder_base, f"fintry_summary_{timestamp}.xlsx")
figures_folder = os.path.join(output_folder_base, f"fintry_figures_{timestamp}")

# Create figures folder if it doesn't exist
os.makedirs(figures_folder, exist_ok=True)



In [3]:
# Step 1: Load dataset
# Description: Read processed EPC dataset with priority indices
df = pd.read_csv(input_file)



In [5]:
# Step 2: Numeric summaries (columns filtered to exist)
numeric_cols = [
    'TOTAL_FLOOR_AREA', 'ENERGY_CONSUMPTION_CURRENT', '3_YR_ENERGY_COST_CURRENT',
    'CO2_EMISS_CURR_PER_FLOOR_AREA', 'SPACE_HEATING_DEMAND', 'WATER_HEATING_DEMAND',
    'NPV', 'IRR', 'COST_PER_TON_CO2_REDUCED', 'PRIORITY_INDEX'  # removed missing columns
]

# Keep only those that exist in the DataFrame
numeric_cols = [col for col in numeric_cols if col in df.columns]

numeric_summary = df[numeric_cols].describe(percentiles=[0.25,0.5,0.75]).T
numeric_summary['missing_count'] = df[numeric_cols].isna().sum()

In [6]:
# Step 3: Categorical summaries
# Description: Frequency counts for key categorical fields
categorical_cols = [
    'CURRENT_ENERGY_RATING', 'POTENTIAL_ENERGY_RATING', 'CURRENT_ENVIRONMENTAL_RATING',
    'POTENTIAL_ENVIRONMENTAL_RATING', 'CONSTRUCTION_AGE_BAND', 'BUILT_FORM',
    'MAIN_HEATING_CATEGORY', 'MAIN_FUEL', 'TENURE'
]
categorical_summary = {col: df[col].value_counts(dropna=False) for col in categorical_cols}



In [12]:
# Step 3a: Create improvement columns if they don't exist
if 'ENERGY_EFFICIENCY_IMPROVEMENT' not in df.columns:
    df['ENERGY_EFFICIENCY_IMPROVEMENT'] = df['POTENTIAL_ENERGY_EFFICIENCY'] - df['CURRENT_ENERGY_EFFICIENCY']

if 'ENVIRONMENT_IMPACT_REDUCTION' not in df.columns:
    df['ENVIRONMENT_IMPACT_REDUCTION'] = df['ENVIRONMENT_IMPACT_POTENTIAL'] - df['ENVIRONMENT_IMPACT_CURRENT']

if 'ENERGY_COST_SAVINGS' not in df.columns:
    df['ENERGY_COST_SAVINGS'] = df['3_YR_ENERGY_SAVINGS_POTENTIAL']

In [13]:
# Step 4: Improvement potential summary
# Description: Summary of energy, CO2, and cost improvement potential

# List of columns we want
improvement_cols = ['ENERGY_EFFICIENCY_IMPROVEMENT', 'ENVIRONMENT_IMPACT_REDUCTION', 'ENERGY_COST_SAVINGS']

# Keep only those that exist
improvement_cols = [col for col in improvement_cols if col in df.columns]

# Compute summary statistics only for existing columns
improvement_summary = df[improvement_cols].agg(['mean','median','max','min','std'])
improvement_summary['missing_count'] = df[improvement_cols].isna().sum()

In [15]:
# Step 5: Measures applicability summary
# Description: Count of homes eligible per improvement measure
measure_cols = [col for col in df.columns if 'IMPROVEMENTS_PARSED_' in col]  # adjust if needed
measures_summary = df[measure_cols].sum().sort_values(ascending=False)



In [16]:
# Step 6: Export tables to Excel (robust version)
with pd.ExcelWriter(output_excel) as writer:

    # Numeric summary
    if 'numeric_summary' in locals() and not numeric_summary.empty:
        numeric_summary.to_excel(writer, sheet_name='Numeric_Summary')
    else:
        pd.DataFrame().to_excel(writer, sheet_name='Numeric_Summary')

    # Improvement summary
    if 'improvement_summary' in locals() and not improvement_summary.empty:
        improvement_summary.to_excel(writer, sheet_name='Improvement_Summary')
    else:
        pd.DataFrame().to_excel(writer, sheet_name='Improvement_Summary')

    # Measures summary
    if 'measures_summary' in locals():
        if isinstance(measures_summary, pd.Series):
            measures_summary.to_frame(name='Eligible_Homes').to_excel(writer, sheet_name='Measures_Summary')
        else:
            measures_summary.to_excel(writer, sheet_name='Measures_Summary')
    else:
        pd.DataFrame().to_excel(writer, sheet_name='Measures_Summary')

    # Categorical summaries
    if 'categorical_summary' in locals():
        for col, counts in categorical_summary.items():
            if counts is not None and not counts.empty:
                safe_sheet_name = f"{col[:25]}_freq"  # truncate to avoid Excel sheet name limit
                counts.to_frame(name='Count').to_excel(writer, sheet_name=safe_sheet_name)

In [17]:
# Step 7: Figures - Numeric distributions
# Description: Create histograms and boxplots for key numeric fields
for col in numeric_cols:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], kde=True, color='skyblue')
    plt.title(f'{col} Distribution')
    plt.tight_layout()
    plt.savefig(os.path.join(figures_folder, f"{col}_hist.png"))
    plt.close()
    
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[col], color='lightgreen')
    plt.title(f'{col} Boxplot')
    plt.tight_layout()
    plt.savefig(os.path.join(figures_folder, f"{col}_boxplot.png"))
    plt.close()



In [18]:
# Step 8: Figures - Categorical counts
# Description: Bar plots for categorical variables
for col in categorical_cols:
    plt.figure(figsize=(8,4))
    sns.countplot(data=df, y=col, order=df[col].value_counts().index, palette='pastel')
    plt.title(f'{col} Counts')
    plt.tight_layout()
    plt.savefig(os.path.join(figures_folder, f"{col}_barplot.png"))
    plt.close()



/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_6158/843273018.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df, y=col, order=df[col].value_counts().index, palette='pastel')
/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_6158/843273018.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df, y=col, order=df[col].value_counts().index, palette='pastel')
/var/folders/57/_mkdbdyx3v15yrmh09sz1jmr0000gn/T/ipykernel_6158/843273018.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df, y=col, order=d

In [19]:
# Step 9: Figures - Improvement vs Cost or Priority
# Description: Scatter plots showing relationships between potential, cost-effectiveness, and priority
plt.figure(figsize=(6,5))
sns.scatterplot(data=df, x='ENERGY_COST_SAVINGS', y='PRIORITY_INDEX', hue='CONSTRUCTION_AGE_BAND', palette='viridis')
plt.title('Energy Cost Savings vs Priority Index')
plt.tight_layout()
plt.savefig(os.path.join(figures_folder, "EnergyCost_vs_PriorityIndex.png"))
plt.close()

plt.figure(figsize=(6,5))
sns.scatterplot(data=df, x='ENVIRONMENT_IMPACT_REDUCTION', y='PRIORITY_INDEX', hue='BUILT_FORM', palette='coolwarm')
plt.title('CO2 Reduction vs Priority Index')
plt.tight_layout()
plt.savefig(os.path.join(figures_folder, "CO2Reduction_vs_PriorityIndex.png"))
plt.close()


In [20]:
# Step 10: Completion message
print(f"All tables exported to: {output_excel}")
print(f"All figures saved in folder: {figures_folder}")

All tables exported to: /Users/jakegehrung/Desktop/data_raw/final_deliverables/fintry_summary_20260305_115416.xlsx
All figures saved in folder: /Users/jakegehrung/Desktop/data_raw/final_deliverables/fintry_figures_20260305_115416
